In [10]:
# !pip install h3
import h3






In [3]:

#Exemplo 1: De (lat, lng) para H3 index

# lat, lng, resolução
index = h3.latlng_to_cell(-23.5505, -46.6333, 9) 

print(index) 
#'89a8100c003ffff'  (São Paulo)

#Agora fazendo resolução 8
index = h3.latlng_to_cell(-23.5505, -46.6333, 8)
print(index)

#Quanto maior o valor da resolução, menor é a área do hexágono.
#Resolução 0: continente 
#Resolução 7: bairro
#Resolução 10: rua



89a8100c02fffff
88a8100c03fffff


In [4]:

#Exemplo 2: De H3 index para (lat, lng)
#Para voltar do H3 index para (lat, lng), podemos usar a função cell_to_latlng.
lat, lng = h3.cell_to_latlng(index)
# → (-23.5505, -46.6333)
print(lat, lng)


-23.552962252442057 -46.63498742338605


In [5]:

#Exemplo 3: Vizinhos de um H3 index

# disco de raio k (centro + todos ao redor)
vizinhos = h3.grid_disk(index, 1)
# → set com 7 células
print(vizinhos)



['88a8100c03fffff', '88a8100c01fffff', '88a8100c07fffff', '88a8100c39fffff', '88a8100c15fffff', '88a8100c1dfffff', '88a8100c0bfffff']


In [6]:

#Exemplo 4: Distância entre duas células em grids 
#Quantos hexágonos existem entre duas células? 
a = h3.latlng_to_cell(-23.55, -46.63, 9)
b = h3.latlng_to_cell(-23.56, -46.62, 9)

dist = h3.grid_distance(a, b)
print(dist)



5


In [ ]:


#Exemplo 5: Subir e descer resoluções
# "subir" para resolução menor (célula pai)
index1 = '89a8100c02fffff'

pai = h3.cell_to_parent(index1, 8)
print(pai)


# "descer" para resolução maior (células filhas)
filhas = h3.cell_to_children(index1, 10)
print(filhas)


#Agora quero "descer" para duas resoluções
filhas = h3.cell_to_children(index1, 11)
print(filhas)


88a8100c03fffff
['8aa8100c02c7fff', '8aa8100c02cffff', '8aa8100c02d7fff', '8aa8100c02dffff', '8aa8100c02e7fff', '8aa8100c02effff', '8aa8100c02f7fff']
['8ba8100c02c0fff', '8ba8100c02c1fff', '8ba8100c02c2fff', '8ba8100c02c3fff', '8ba8100c02c4fff', '8ba8100c02c5fff', '8ba8100c02c6fff', '8ba8100c02c8fff', '8ba8100c02c9fff', '8ba8100c02cafff', '8ba8100c02cbfff', '8ba8100c02ccfff', '8ba8100c02cdfff', '8ba8100c02cefff', '8ba8100c02d0fff', '8ba8100c02d1fff', '8ba8100c02d2fff', '8ba8100c02d3fff', '8ba8100c02d4fff', '8ba8100c02d5fff', '8ba8100c02d6fff', '8ba8100c02d8fff', '8ba8100c02d9fff', '8ba8100c02dafff', '8ba8100c02dbfff', '8ba8100c02dcfff', '8ba8100c02ddfff', '8ba8100c02defff', '8ba8100c02e0fff', '8ba8100c02e1fff', '8ba8100c02e2fff', '8ba8100c02e3fff', '8ba8100c02e4fff', '8ba8100c02e5fff', '8ba8100c02e6fff', '8ba8100c02e8fff', '8ba8100c02e9fff', '8ba8100c02eafff', '8ba8100c02ebfff', '8ba8100c02ecfff', '8ba8100c02edfff', '8ba8100c02eefff', '8ba8100c02f0fff', '8ba8100c02f1fff', '8ba8100c02f2

In [ ]:

#Visualizar 
import h3
import folium
import json

# 1. gerar hexágonos ao redor de São Paulo
centro = (-23.5505, -46.6333)
index  = h3.latlng_to_cell(*centro, 9)
celulas = h3.grid_disk(index, 3)   # disco de raio 3

# 2. converter para GeoJSON
features = []
for cell in celulas:
    boundary = h3.cell_to_boundary(cell)
    coords   = [[lng, lat] for lat, lng in boundary]
    coords.append(coords[0])          # fechar o polígono
    features.append({
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [coords]},
        "properties": {"h3_index": cell},
    })
geojson = {"type": "FeatureCollection", "features": features}

# 3. criar o mapa
m = folium.Map(location=centro, zoom_start=13, tiles="CartoDB positron")

folium.GeoJson(
    geojson,
    style_function=lambda f: {
        "fillColor": "#4A90D9",
        "color": "#1A5276",
        "weight": 1,
        "fillOpacity": 0.4,
    },
    tooltip=folium.GeoJsonTooltip(fields=["h3_index"]),
).add_to(m)

m.save("hexagonos.html")   

c:\Quarto\VRP_OR_TOOLS\Codigo_apoio\.venv\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
